In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
import duckdb

In [7]:
items_path = '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv'

# Query 1: Total Orders & Total Revenue
q1 = f"""
SELECT 
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS total_revenue
FROM '{items_path}'
"""

duckdb.query(q1).df()

,total_orders,total_revenue
0,98666,13591643.7


In [8]:
products_path = '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv'

# Query 2: Top 5 Product Categories
q2 = f"""
SELECT 
    p.product_category_name AS category_name,
    ROUND(SUM(i.price), 2) AS total_revenue
FROM '{items_path}' i
JOIN '{products_path}' p ON i.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY category_name
ORDER BY total_revenue DESC
LIMIT 5
"""

duckdb.query(q2).df()

,category_name,total_revenue
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32


In [9]:
payments_path = '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv'

# Query 3: Payment Type Popularity & Average Order Value
q3 = f"""
SELECT 
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(payment_value), 2) AS avg_order_value
FROM '{payments_path}'
GROUP BY payment_type
ORDER BY total_orders DESC
"""

duckdb.query(q3).df()

,payment_type,total_orders,avg_order_value
0,credit_card,76505,163.32
1,boleto,19784,145.03
2,voucher,3866,65.70
3,debit_card,1528,142.57
4,not_defined,3,0.00


In [10]:
orders_path = '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv'

# Query 4: Month-over-Month (MoM) Growth Rate
q4 = f"""
WITH monthly_sales AS (
    SELECT 
        DATE_TRUNC('month', CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS month,
        ROUND(SUM(i.price), 2) AS total_revenue
    FROM '{orders_path}' o
    JOIN '{items_path}' i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
)
SELECT 
    STRFTIME(month, '%Y-%m') AS sales_month,
    total_revenue,
    LAG(total_revenue) OVER (ORDER BY month) AS prev_month_revenue,
    ROUND(((total_revenue - LAG(total_revenue) OVER (ORDER BY month)) / LAG(total_revenue) OVER (ORDER BY month)) * 100, 2) AS mom_growth_pct
FROM monthly_sales
ORDER BY month
"""

duckdb.query(q4).df()

,sales_month,total_revenue,prev_month_revenue,mom_growth_pct
0,2016-09,134.97,NaN,NaN
1,2016-10,40325.11,134.97,29777.09
2,2016-12,10.90,40325.11,-99.97
3,2017-01,111798.36,10.90,1025573.03
4,2017-02,234223.40,111798.36,109.51
5,2017-03,359198.85,234223.40,53.36
6,2017-04,340669.68,359198.85,-5.16
7,2017-05,489338.25,340669.68,43.64
8,2017-06,421923.37,489338.25,-13.78
9,2017-07,481604.52,421923.37,14.15


In [11]:
# Query 5: Delivery Days & Late Delivery Rate
q5 = f"""
SELECT 
    ROUND(AVG(DATE_DIFF('day', CAST(order_purchase_timestamp AS TIMESTAMP), CAST(order_delivered_customer_date AS TIMESTAMP))), 1) AS avg_delivery_days,
    ROUND(COUNT(CASE WHEN CAST(order_delivered_customer_date AS TIMESTAMP) > CAST(order_estimated_delivery_date AS TIMESTAMP) THEN 1 END) * 100.0 / COUNT(order_id), 2) AS late_delivery_percentage
FROM '{orders_path}'
WHERE order_status = 'delivered'
"""

duckdb.query(q5).df()

,avg_delivery_days,late_delivery_percentage
0,12.5,8.11


In [3]:
# Query 6: Top 10 Revenue-Generating States
q6 = """
SELECT 
    c.customer_state,
    ROUND(SUM(i.price), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv' o
JOIN '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv' i 
  ON o.order_id = i.order_id
JOIN '/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv' c 
  ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

duckdb.query(q6).df()

,customer_state,total_revenue,total_orders
0,SP,5067633.16,40501
1,RJ,1759651.13,12350
2,MG,1552481.83,11354
3,RS,728897.47,5345
4,PR,666063.51,4923
5,SC,507012.13,3546
6,BA,493584.14,3256
7,DF,296498.41,2080
8,GO,282836.70,1957
9,ES,268643.45,1995
